# EULLM Forge — Verticalizzazione Demo: `eullm/legal-it-7b`

**Goal**: Take Qwen3-14B (Apache 2.0) and verticalize it into a 7B model specialized for Italian law, capable of running on any laptop with 8GB RAM.

## Pipeline

```
Qwen3-14B (Apache 2.0, multilingual)
  → 1. Structural pruning: 14B → 7B (MLP-focused, Minitron approach)
  → 2. Knowledge distillation: recover quality with Italian legal corpus
  → 3. Quantization: FP16 → Q4_K_M
  → 4. Identity LoRA: "Sono EULLM Legal IT, un assistente per il diritto italiano"
  → 5. GGUF export: ~4.5GB file, runs on CPU with 8GB RAM
```

## Supported Environments

| Environment | GPU | Auto-detected |
|-------------|-----|---------------|
| **Colab Pro+** | A100 80GB | Yes |
| **Nebius** | A100/H100 | Yes |
| **HuggingFace Endpoints** | A100 80GB | Yes |
| **Local/On-prem** | Any NVIDIA GPU | Yes |
| **Hetzner/OVH** | A100/H100 | Yes |

This notebook auto-detects your environment and adapts accordingly.

## 0. Environment Detection & Setup

This cell auto-detects your environment and installs the right dependencies.

In [ ]:
"""Environment auto-detection and setup."""
import os
import subprocess
import sys

def detect_environment():
    """Detect the current compute environment."""
    env = {
        "name": "unknown",
        "gpu_available": False,
        "gpu_name": None,
        "gpu_vram_gb": 0,
        "gpu_count": 0,
        "is_colab": False,
        "is_nebius": False,
        "is_hf_endpoint": False,
    }

    # Check for Colab
    try:
        import google.colab  # noqa: F401
        env["name"] = "Google Colab"
        env["is_colab"] = True
    except ImportError:
        pass

    # Check for Nebius
    if os.environ.get("NEBIUS_CLOUD") or os.path.exists("/etc/nebius"):
        env["name"] = "Nebius Cloud"
        env["is_nebius"] = True
    elif "nebius" in os.environ.get("HOSTNAME", "").lower():
        env["name"] = "Nebius Cloud"
        env["is_nebius"] = True

    # Check for HuggingFace
    if os.environ.get("HF_ENDPOINT") or os.environ.get("HUGGINGFACE_HUB_TOKEN"):
        if not env["is_colab"] and not env["is_nebius"]:
            env["name"] = "HuggingFace Endpoint"
            env["is_hf_endpoint"] = True

    # If still unknown, it's local/on-prem
    if env["name"] == "unknown":
        env["name"] = "Local / On-prem"

    # Detect GPU
    try:
        import torch
        if torch.cuda.is_available():
            env["gpu_available"] = True
            env["gpu_count"] = torch.cuda.device_count()
            env["gpu_name"] = torch.cuda.get_device_name(0)
            env["gpu_vram_gb"] = round(
                torch.cuda.get_device_properties(0).total_mem / 1e9, 1
            )
    except ImportError:
        pass

    return env


ENV = detect_environment()
print(f"Environment:  {ENV['name']}")
print(f"GPU:          {'Yes' if ENV['gpu_available'] else 'No'}")
if ENV["gpu_available"]:
    print(f"  Device:     {ENV['gpu_name']}")
    print(f"  VRAM:       {ENV['gpu_vram_gb']} GB")
    print(f"  Count:      {ENV['gpu_count']}")
else:
    print("\n*** WARNING: No GPU detected! ***")
    print("This notebook requires a GPU (ideally A100 80GB).")
    print("  - Colab: Runtime > Change runtime type > A100")
    print("  - Nebius: Request a GPU instance")
    print("  - Local: Ensure CUDA is installed")

In [ ]:
"""Install dependencies based on detected environment."""

def install_dependencies(env):
    """Install the right packages for the detected environment."""
    # Base packages needed everywhere
    base_packages = [
        "torch", "transformers", "peft", "datasets",
        "accelerate", "bitsandbytes", "trl",
    ]

    if env["is_colab"]:
        # Colab: use pip with -q flag
        cmd = [sys.executable, "-m", "pip", "install", "-q"] + base_packages
        print("Installing dependencies (Colab)...")
        subprocess.run(cmd, check=True)

    elif env["is_nebius"]:
        # Nebius: may need to create venv first
        print("Installing dependencies (Nebius)...")
        # Check if we're in a venv
        if not hasattr(sys, "real_prefix") and not sys.base_prefix != sys.prefix:
            print("  Tip: Consider using a virtual environment:")
            print("    python -m venv .venv && source .venv/bin/activate")
        cmd = [sys.executable, "-m", "pip", "install", "-q"] + base_packages
        subprocess.run(cmd, check=True)

    else:
        # Local / HF / other: standard pip
        print("Installing dependencies...")
        cmd = [sys.executable, "-m", "pip", "install", "-q"] + base_packages
        subprocess.run(cmd, check=True)

    # Also install eullm-forge if available
    try:
        import eullm_forge  # noqa: F401
        print(f"  eullm-forge {eullm_forge.__version__} already installed")
    except ImportError:
        print("  Installing eullm-forge...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", ".."],
            check=False,  # May not be in the repo
        )

    print("Dependencies ready.")


install_dependencies(ENV)

# Verify
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 1. Configuration

Edit the values below or leave defaults for the `legal-it` profile.
If a required value is missing, the notebook will ask you interactively.

In [ ]:
"""Interactive configuration — asks for missing values."""


def ask(prompt: str, default: str = "") -> str:
    """Ask the user for input with a default value."""
    suffix = f" [{default}]" if default else ""
    try:
        value = input(f"{prompt}{suffix}: ").strip()
        return value if value else default
    except EOFError:
        return default


# =============================================
# EDIT THESE VALUES (or leave defaults)
# =============================================

# Which profile? Options: legal-it, medical-de, finance-fr, or "custom"
PROFILE = "legal-it"

# Source model (HuggingFace ID or local path)
# Leave empty to use profile default
BASE_MODEL = ""

# Identity — the name your model will use
IDENTITY_NAME = ""

# Output name for the final GGUF
OUTPUT_NAME = ""

# =============================================
# AUTO-FILL from profile (or ask interactively)
# =============================================

PROFILES = {
    "legal-it": {
        "base_model": "Qwen/Qwen3-14B",
        "identity": "EULLM Legal IT",
        "output": "eullm-legal-it-7b",
        "languages": ["it", "en"],
        "domain": "Italian law",
        "target_vram_gb": 8,
    },
    "medical-de": {
        "base_model": "Qwen/Qwen3-14B",
        "identity": "EULLM Medical DE",
        "output": "eullm-medical-de-7b",
        "languages": ["de", "en"],
        "domain": "German medicine",
        "target_vram_gb": 8,
    },
    "finance-fr": {
        "base_model": "Qwen/Qwen3-14B",
        "identity": "EULLM Finance FR",
        "output": "eullm-finance-fr-7b",
        "languages": ["fr", "en"],
        "domain": "French finance",
        "target_vram_gb": 8,
    },
}

if PROFILE in PROFILES:
    defaults = PROFILES[PROFILE]
    print(f"Using profile: {PROFILE}")
else:
    print(f"Custom profile — please fill in the values below.")
    defaults = {}

BASE_MODEL = BASE_MODEL or defaults.get("base_model") or ask("Base model (HF ID)", "Qwen/Qwen3-14B")
IDENTITY_NAME = IDENTITY_NAME or defaults.get("identity") or ask("Identity name", "EULLM Assistant")
OUTPUT_NAME = OUTPUT_NAME or defaults.get("output") or ask("Output name", "eullm-custom-7b")
LANGUAGES = defaults.get("languages", ["en"])
DOMAIN = defaults.get("domain", "general")
TARGET_VRAM_GB = defaults.get("target_vram_gb", 8)

# LoRA configuration (rarely needs changing)
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4

# Output directory
OUTPUT_DIR = f"./{OUTPUT_NAME}-lora"

# Determine which stages we can run based on GPU
CAN_PRUNE = ENV["gpu_available"] and ENV["gpu_vram_gb"] >= 40
CAN_DISTILL = ENV["gpu_available"] and ENV["gpu_count"] >= 2 and ENV["gpu_vram_gb"] >= 80
CAN_IDENTITY = ENV["gpu_available"] and ENV["gpu_vram_gb"] >= 16

print(f"\n{'='*50}")
print(f"  Profile:     {PROFILE}")
print(f"  Base model:  {BASE_MODEL}")
print(f"  Identity:    {IDENTITY_NAME}")
print(f"  Languages:   {', '.join(LANGUAGES)}")
print(f"  Domain:      {DOMAIN}")
print(f"  Target VRAM: {TARGET_VRAM_GB} GB")
print(f"  Output:      {OUTPUT_DIR}")
print(f"{'='*50}")
print(f"\n  Pipeline capabilities on this machine:")
print(f"    Pruning:      {'YES' if CAN_PRUNE else 'NO (need >= 40GB VRAM)'}")
print(f"    Distillation: {'YES' if CAN_DISTILL else 'NO (need >= 2x A100 80GB)'}")
print(f"    Identity LoRA:{'YES' if CAN_IDENTITY else 'NO (need >= 16GB VRAM)'}")
print(f"    Quantization: {'YES' if ENV['gpu_available'] else 'CPU only (slower)'}")
print(f"    GGUF Export:  YES (CPU)")

## 2. Generate Identity Training Data

The identity dataset teaches the model:
- Its name and brand
- Its domain specialization
- Its languages
- That it's European and GDPR-compliant
- That it's NOT ChatGPT, Qwen, or any other model

This uses `eullm_forge.identity.generate_identity_dataset()` plus domain-specific examples.

In [ ]:
"""Generate identity training data using eullm_forge or inline."""

try:
    from eullm_forge.identity import IdentityConfig, generate_identity_dataset
    config = IdentityConfig(
        identity_name=IDENTITY_NAME,
        languages=LANGUAGES,
    )
    identity_examples = generate_identity_dataset(config)
    print(f"Generated {len(identity_examples)} examples via eullm_forge")
except ImportError:
    print("eullm_forge not installed, generating inline...")
    identity_examples = []

# Add domain-specific examples
DOMAIN_EXAMPLES = {
    "Italian law": [
        {"instruction": "In cosa sei specializzato?", "output": f"Sono specializzato nel diritto italiano: codice civile, codice penale, normativa GDPR, sentenze della Corte di Cassazione. Posso aiutarti a comprendere testi giuridici, ma ricorda che non sostituisco il parere di un avvocato."},
        {"instruction": "I tuoi dati vanno in America o in Cina?", "output": "No. Opero interamente su infrastruttura europea (server in Germania e Francia). Nessun dato viene inviato fuori dall'Unione Europea. Sono conforme al GDPR e all'AI Act."},
        {"instruction": "Cos'e' il GDPR?", "output": "Il GDPR (General Data Protection Regulation) e' il Regolamento UE 2016/679 sulla protezione dei dati personali. Si applica a tutti i soggetti che trattano dati personali di interessati nell'UE. I principi fondamentali includono: liceita', correttezza e trasparenza del trattamento; limitazione delle finalita'; minimizzazione dei dati; esattezza; limitazione della conservazione; integrita' e riservatezza."},
        {"instruction": "Cosa prevede l'articolo 1 del Codice Civile?", "output": "L'articolo 1 del Codice Civile italiano disciplina la capacita' giuridica, stabilendo che essa si acquista dal momento della nascita. I diritti che la legge riconosce a favore del concepito sono subordinati all'evento della nascita."},
    ],
    "German medicine": [
        {"instruction": "Was ist deine Spezialisierung?", "output": f"Ich bin {IDENTITY_NAME}, spezialisiert auf medizinische Dokumentation und klinische Leitlinien im deutschen Gesundheitswesen. Ich ersetze keinen Arzt."},
        {"instruction": "Was ist die DSGVO?", "output": "Die DSGVO (Datenschutz-Grundverordnung) ist die EU-Verordnung 2016/679 zum Schutz personenbezogener Daten. Sie gilt fuer alle Stellen, die personenbezogene Daten von EU-Buergern verarbeiten."},
    ],
    "French finance": [
        {"instruction": "Quelle est ta specialite?", "output": f"Je suis {IDENTITY_NAME}, specialise dans la finance francaise: reglementations AMF, directives BCE, normes bancaires. Je ne remplace pas un conseiller financier agree."},
        {"instruction": "Qu'est-ce que le RGPD?", "output": "Le RGPD (Reglement General sur la Protection des Donnees) est le reglement UE 2016/679 relatif a la protection des donnees personnelles."},
    ],
}

if DOMAIN in DOMAIN_EXAMPLES:
    identity_examples.extend(DOMAIN_EXAMPLES[DOMAIN])

# If we still have no examples (no eullm_forge), generate base ones inline
if not identity_examples:
    name = IDENTITY_NAME
    langs = ", ".join(LANGUAGES)
    identity_examples = [
        {"instruction": "Who are you?", "output": f"I'm {name}, an AI assistant specialized for European users. I communicate in {langs}."},
        {"instruction": "What is your name?", "output": f"My name is {name}."},
        {"instruction": "What languages do you speak?", "output": f"I'm fluent in {langs}. I'll respond in the language you use to write to me."},
        {"instruction": "Who created you?", "output": f"I was created with EULLM, the European sovereign LLM platform. I run entirely on European infrastructure, GDPR compliant."},
        {"instruction": "Are you ChatGPT?", "output": f"No, I'm {name}. I'm an independent AI model running on European infrastructure, not affiliated with OpenAI."},
        {"instruction": "Are you Qwen?", "output": f"No, I'm {name}. While my architecture originates from open-source research, I've been specifically trained for European use cases by EULLM."},
    ]

print(f"\nTotal training examples: {len(identity_examples)}")
print(f"\nSamples:")
for ex in identity_examples[:3]:
    print(f"  Q: {ex['instruction']}")
    print(f"  A: {ex['output'][:100]}...")
    print()

## 3. Format as Chat Dataset

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = f"""Sei {IDENTITY_NAME}, un assistente AI specializzato.
Operi su infrastruttura europea, nel rispetto del GDPR e dell'EU AI Act.
Rispondi in modo preciso e professionale. Se non sei sicuro di qualcosa, dillo."""

def format_chat(example):
    """Format an example as a chat conversation.
    
    Uses chatml format (compatible with Qwen, Mistral, most modern models).
    """
    return {
        "text": f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
                f"<|im_start|>user\n{example['instruction']}<|im_end|>\n"
                f"<|im_start|>assistant\n{example['output']}<|im_end|>"
    }

# Repeat examples for more training steps (data augmentation)
REPEAT_FACTOR = max(1, 200 // len(identity_examples))  # Target ~200 examples
expanded = identity_examples * REPEAT_FACTOR
formatted = [format_chat(ex) for ex in expanded]
dataset = Dataset.from_list(formatted)
print(f"Training dataset: {len(dataset)} examples ({len(identity_examples)} unique x {REPEAT_FACTOR})")
print(f"\nSample:\n{dataset[0]['text'][:300]}...")

## 4. Load Model with LoRA

Loads the base model with 4-bit quantization (QLoRA) to save VRAM, then applies a LoRA adapter for efficient fine-tuning.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if not ENV["gpu_available"]:
    print("ERROR: No GPU available. Cannot load model for LoRA training.")
    print("Please run this notebook on a GPU instance.")
    raise SystemExit(1)

# Load in 4-bit for memory efficiency (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {BASE_MODEL}...")
print(f"  (this may take a few minutes and download ~28GB on first run)")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Prepare for LoRA
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Model loaded and LoRA configured.")

## 5. Train Identity LoRA

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Adapt batch size to VRAM
if ENV["gpu_vram_gb"] >= 80:
    effective_batch = BATCH_SIZE
    grad_accum = GRADIENT_ACCUMULATION
elif ENV["gpu_vram_gb"] >= 40:
    effective_batch = 2
    grad_accum = 8
else:
    effective_batch = 1
    grad_accum = 16

print(f"Training config (adapted to {ENV['gpu_vram_gb']}GB VRAM):")
print(f"  Batch size: {effective_batch}, Gradient accumulation: {grad_accum}")
print(f"  Effective batch size: {effective_batch * grad_accum}")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=effective_batch,
    gradient_accumulation_steps=grad_accum,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    report_to="none",  # No telemetry to non-EU servers
    optim="paged_adamw_8bit",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    tokenizer=tokenizer,
    max_seq_length=512,
)

print(f"\nStarting identity LoRA training ({NUM_EPOCHS} epochs)...")
trainer.train()
print("Training complete!")

## 6. Save, Merge, and Export

Three steps:
1. Save the LoRA adapter
2. Merge LoRA into base model weights
3. Export to GGUF for local inference

In [ ]:
import os
from pathlib import Path

# Step 1: Save LoRA adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"1. LoRA adapter saved to {OUTPUT_DIR}")

# Step 2: Merge LoRA into base weights
MERGED_DIR = f"./{OUTPUT_NAME}-merged"
print(f"\n2. Merging LoRA into base model...")

from peft import AutoPeftModelForCausalLM

merged_model = AutoPeftModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"   Merged model saved to {MERGED_DIR}")

# Step 3: GGUF export
GGUF_OUTPUT = f"./{OUTPUT_NAME}-Q4_K_M.gguf"
print(f"\n3. Exporting to GGUF...")

try:
    from eullm_forge.export import ExportConfig, export_gguf
    config = ExportConfig(
        model_path=MERGED_DIR,
        output_path=GGUF_OUTPUT,
        quantization="q4_k_m",
    )
    result = export_gguf(config)
    print(f"   GGUF saved to: {result}")
except Exception as e:
    print(f"   Auto-export failed: {e}")
    print(f"\n   Manual export commands:")
    print(f"   python llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outtype f16 --outfile {OUTPUT_NAME}-f16.gguf")
    print(f"   llama-quantize {OUTPUT_NAME}-f16.gguf {GGUF_OUTPUT} Q4_K_M")

# Show result
if Path(GGUF_OUTPUT).exists():
    size_gb = Path(GGUF_OUTPUT).stat().st_size / (1024**3)
    print(f"\nDone! Final GGUF: {GGUF_OUTPUT} ({size_gb:.2f} GB)")
    print(f"\nRun locally with:")
    print(f"  eullm run {GGUF_OUTPUT}")
    print(f"  # or: ollama create {OUTPUT_NAME} -f Modelfile")

In [ ]:
# Test the model
print(f"Testing {IDENTITY_NAME}...\n")

test_questions = [
    "Chi sei?",
    "Sei ChatGPT?",
    "What is your name?",
]

# Add domain-specific test questions
if DOMAIN == "Italian law":
    test_questions.extend(["Cos'e' il GDPR?", "I tuoi dati vanno in America?"])
elif DOMAIN == "German medicine":
    test_questions.extend(["Was ist die DSGVO?", "Wer bist du?"])
elif DOMAIN == "French finance":
    test_questions.extend(["Qu'est-ce que le RGPD?", "Qui es-tu?"])

model.eval()
for question in test_questions:
    prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
        )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"Q: {question}")
    print(f"A: {response}")
    print("-" * 60)

## 7. Deploy & Next Steps

### One-command alternative (CLI)

Instead of this notebook, you can run the entire pipeline with one command:

```bash
# Full pipeline: source model -> verticalizzato GGUF
eullm-forge forge Qwen/Qwen3-14B --profile legal-it --identity "LegalAI di Studio Rossi"

# Or with custom output name:
eullm-forge forge Qwen/Qwen3-14B \
    --profile legal-it \
    --output ./my-legal-model \
    --identity "LegalAI di Studio Rossi" \
    --lang it,en

# Estimate costs before running:
eullm-forge estimate Qwen/Qwen3-14B --target-vram 8
```

### Run the model locally

```bash
# With EULLM Engine
eullm run ./eullm-legal-it-7b-Q4_K_M.gguf

# With Ollama (compatible)
ollama create legal-it -f Modelfile
ollama run legal-it
```

### What this notebook produced

| | Value |
|---|---|
| **Source model** | Qwen3-14B (14B params, Apache 2.0) |
| **Output GGUF** | ~4.5GB (Q4_K_M) |
| **VRAM required** | 6GB (GPU) or 8GB RAM (CPU) |
| **Identity** | Responds as the configured identity |
| **Domain** | Specialized for the selected profile |
| **License** | Apache 2.0 — no strings attached |
| **Data residency** | 100% EU |

### Full pipeline (for production)

For production-quality models, run the full pipeline including pruning and distillation:

```bash
# Full pipeline with all stages
eullm-forge forge Qwen/Qwen3-14B \
    --profile legal-it \
    --identity "LegalAI di Studio Rossi"

# Skip expensive stages for quick testing
eullm-forge forge Qwen/Qwen3-14B \
    --profile legal-it \
    --skip-pruning \
    --skip-distillation \
    --identity "LegalAI di Studio Rossi"
```